In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# ==========================================
# 1. 定义极其简单的 Toy 模型参数
# ==========================================
in_features = 4
out_features = 2
original_size = 8
compressed_size = 3  # 把 8 个权重压缩到 3 个格子里
num_rows = 2         # 使用 2 行哈希映射

# 我们手工定义 8 个具有代表性的权重（包含大数和小数，正数和负数）
# 真实权重参数 (需要被更新的参数)
weight = nn.Parameter(torch.tensor([
    [ 0.1, -5.0,  3.0, -0.2],  # 第一行输出的权重
    [-8.0,  0.5, -0.1,  9.0]   # 第二行输出的权重
], requires_grad=True))

# 手工定义哈希映射索引 (故意制造冲突)
# 值域为 0, 1, 2 (对应 compressed_size=3)
hash_indices = torch.tensor([
    [0, 0, 1, 1, 2, 2, 0, 1], # 第 0 行草图的映射规则 (比如 0.1 和 -5.0 都映射到格子 0)
    [1, 2, 0, 2, 1, 0, 1, 2]  # 第 1 行草图的映射规则
])

# 随便定义一个输入样本 (Batch=1, features=4)
x = torch.tensor([[1.0, 1.0, 1.0, 1.0]])
# 定义分类目标 (二分类)
target = torch.tensor([1])

print("==== 初始状态 ====")
print(f"原始真实权重:\n{weight.data}\n")
print(f"哈希映射规则 (2行 x 8个权重):\n{hash_indices}\n")



==== 初始状态 ====
原始真实权重:
tensor([[ 0.1000, -5.0000,  3.0000, -0.2000],
        [-8.0000,  0.5000, -0.1000,  9.0000]])

哈希映射规则 (2行 x 8个权重):
tensor([[0, 0, 1, 1, 2, 2, 0, 1],
        [1, 2, 0, 2, 1, 0, 1, 2]])



In [ ]:
# ==========================================
# 2. 前向传播：Fake Compression (假压缩) 过程
# ==========================================
print("==== 前向传播: AbsMaxMin 假压缩 ====")
weight_flat = weight.view(-1)
# flat后变成: [0.1, -5.0, 3.0, -0.2, -8.0, 0.5, -0.1, 9.0]

# (A) 模拟 AbsMin (绝对值最小) 存入草图
abs_w = torch.abs(weight_flat)
# 降序排列：绝对值大的先存，绝对值小的后存（会覆盖大的），实现 AbsMin 效果
sort_idx = torch.argsort(abs_w, descending=True)
sorted_w = weight_flat[sort_idx]
sorted_indices = hash_indices[:, sort_idx]

# 初始化草图为正无穷
sketch_states = torch.full((num_rows, compressed_size), float('inf'))
# 写入草图
sorted_w_expanded = sorted_w.unsqueeze(0).expand(num_rows, -1)
sketch_states.scatter_(1, sorted_indices, sorted_w_expanded)

print(f"经过 AbsMin 散布后的多行草图状态 (Sketch States):\n{sketch_states}\n")

# (B) 模拟 Max 检索解压
gathered = torch.gather(sketch_states, 1, hash_indices)
print(f"按哈希索引从草图中取出的候选值:\n{gathered}\n")

w_compressed_flat, _ = torch.max(gathered, dim=0)
w_compressed = w_compressed_flat.view(out_features, in_features)
print(f"执行 Max 操作后最终解压的权重:\n{w_compressed}\n")



In [ ]:
# ==========================================
# 3. 前向传播：使用 STE (直通估计器) 构建计算图
# ==========================================
print("==== 前向传播: STE 梯度截断与图拼接 ====")
# 【STE 核心代码】
# 前向计算时: detach() 的部分相抵消，值等于 w_compressed
# 反向传播时: detach() 的部分梯度为 0，只有 + weight 这一项起作用，梯度直接传给原始 weight
w_ste = w_compressed.detach() - weight.detach() + weight

print("用于最终计算线性层的权重 (数值上等于压缩后的权重):")
print(w_ste.data)

# 进行网络前向输出
logits = F.linear(x, w_ste)
print(f"\n模型预测 Logits: {logits.data}")



In [ ]:
# ==========================================
# 4. 反向传播：计算 Loss 并验证 STE
# ==========================================
print("\n==== 反向传播: 验证 STE 是否生效 ====")
loss = F.cross_entropy(logits, target)
print(f"计算得到的 Loss: {loss.item():.4f}")

# 开始反向传播
loss.backward()

print("\n模型原始权重的梯度 (weight.grad):")
print(weight.grad)
print("\n我们可以看到，即便经历了不可导的哈希、覆盖、求最大值等离散操作，")
print("STE依然强行劈开了一条通路，把 Loss 产生的梯度完美传回给了【原始真实权重】！")